PART 1: THE DATASET

1.We want the computer to guess the price of a plane ticket before it is sold.

(regression)

2. What affects airline ticket prices?

3. Dataset Features (What the computer sees)
Column	Meaning (Child Version)
airline	Name of airline
origin	Where the plane starts
destination	Where it goes
distance_km	How far it flies
days_to_departure	How many days before flight
departure_hour	Time of day
is_weekend	Weekend or not
demand_index	How popular the flight is
fuel_price_index	Fuel cost level
ticket_price	What we want to predict

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 3000

airlines = ["Kenya Airways", "Ethiopian Airlines", "EgyptAir"]
routes = [
    ("Nairobi", "Mombasa", 440),
    ("Nairobi", "Johannesburg", 2900),
    ("Addis Ababa", "Lagos", 3900),
    ("Cairo", "Accra", 3300),
    ("Nairobi", "Dubai", 3500)
]

data = []

for _ in range(n):
    airline = np.random.choice(airlines)
    origin, destination, distance = routes[np.random.randint(len(routes))]
    days_to_departure = np.random.randint(1, 180)
    departure_hour = np.random.randint(0, 24)
    is_weekend = np.random.choice([0, 1])
    demand_index = np.random.uniform(0.8, 1.5)
    fuel_price_index = np.random.uniform(0.9, 1.4)

    base_price = distance * 0.12
    urgency_factor = (180 - days_to_departure) * 1.8
    weekend_factor = 80 if is_weekend else 0

    ticket_price = (
        base_price +
        urgency_factor +
        weekend_factor
    ) * demand_index * fuel_price_index

    data.append([
        airline, origin, destination, distance,
        days_to_departure, departure_hour,
        is_weekend, demand_index, fuel_price_index,
        round(ticket_price, 2)
    ])

df = pd.DataFrame(data, columns=[
    "airline", "origin", "destination", "distance_km",
    "days_to_departure", "departure_hour",
    "is_weekend", "demand_index", "fuel_price_index",
    "ticket_price"
])

df.to_csv("african_airline_ticket_prices.csv", index=False)
df.head()


,airline,origin,destination,distance_km,days_to_departure,departure_hour,is_weekend,demand_index,fuel_price_index,ticket_price
0,EgyptAir,Cairo,Accra,3300,93,14,0,1.345784,1.198425,891.24
1,Ethiopian Airlines,Addis Ababa,Lagos,3900,75,10,1,1.033596,0.971433,740.00
2,EgyptAir,Nairobi,Dubai,3500,2,23,1,1.456987,0.900389,1076.25
3,Kenya Airways,Nairobi,Mombasa,440,58,21,0,1.102362,1.045615,313.98
4,EgyptAir,Nairobi,Johannesburg,2900,15,14,1,0.939772,1.157117,788.38


PART 2: MACHINE LEARNING STEP-BY-STEP 🧠
Step 1: Load the data

In [2]:
df = pd.read_csv("african_airline_ticket_prices.csv")


Step 2: Prepare the data (Explain like a child)

Computers do not understand words, only numbers.

So:

Turn airline names into numbers

Turn cities into numbers

In [3]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

X = df.drop("ticket_price", axis=1)
y = df["ticket_price"]

categorical = ["airline", "origin", "destination"]
numerical = [
    "distance_km", "days_to_departure",
    "departure_hour", "is_weekend",
    "demand_index", "fuel_price_index"
]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", "passthrough", numerical)
])


Step 3: Split the data

We teach the computer using some data
We test if it learned using new data

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Step 4: Train the model 🎓

We use Random Forest (very good for pricing problems).

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        random_state=42
    ))
])

model.fit(X_train, y_train)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['airline', 'origin',
                                                   'destination']),
                                                 ('num', 'passthrough',
                                                  ['distance_km',
                                                   'days_to_departure',
                                                   'departure_hour',
                                                   'is_weekend', 'demand_index',
                                                   'fuel_price_index'])])),
                ('rf',
                 RandomForestRegressor(max_depth=15, n_estimators=300,
                                       random_state=42))])

Step 5: Check how smart the model is 🧪

In [6]:
from sklearn.metrics import mean_absolute_error, r2_score

preds = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R²:", r2_score(y_test, preds))


MAE: 31.765882034117887
R²: 0.978464843821187


Expected:

MAE ≈ 20–40 USD

R² > 0.90 (very good)

PART 3: SAVE MODEL FOR PRODUCTION 💾

In [7]:
import joblib
joblib.dump(model, "airline_price_model.pkl")


['airline_price_model.pkl']

PART 4: DEPLOYMENT (REAL PRODUCTION APP) 🚀

We’ll use Streamlit (industry-standard for ML demos & internal tools).

In [8]:
pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 55.4 MB/s eta 0:00:00


In [9]:
!pip install streamlit pyngrok


In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

model = joblib.load("airline_price_model.pkl")

st.title("✈️ African Airline Ticket Price Predictor")

airline = st.selectbox("Airline", ["Kenya Airways", "Ethiopian Airlines", "EgyptAir"])
origin = st.selectbox("Origin", ["Nairobi", "Addis Ababa", "Cairo"])
destination = st.selectbox("Destination", ["Mombasa", "Johannesburg", "Lagos", "Accra", "Dubai"])

distance = st.slider("Distance (km)", 300, 4500, 1500)
days = st.slider("Days to Departure", 1, 180, 30)
hour = st.slider("Departure Hour", 0, 23, 10)
weekend = st.selectbox("Weekend?", [0, 1])
demand = st.slider("Demand Level", 0.8, 1.5, 1.0)
fuel = st.slider("Fuel Price Index", 0.9, 1.4, 1.0)

if st.button("Predict Price"):
    input_df = pd.DataFrame([{
        "airline": airline,
        "origin": origin,
        "destination": destination,
        "distance_km": distance,
        "days_to_departure": days,
        "departure_hour": hour,
        "is_weekend": weekend,
        "demand_index": demand,
        "fuel_price_index": fuel
    }])

    price = model.predict(input_df)[0]
    st.success(f"Estimated Ticket Price: USD {price:.2f}")


Writing app.py


LAUNCH THE aPP

In [11]:
!streamlit run app.py &>/content/logs.txt &


In [12]:
from pyngrok import ngrok

ngrok.set_auth_token("398ykxnqWRpkaexYNblZgOPHLue_6KHcYD3Tm5CmVDHnBtcEg")


In [13]:
!streamlit run app.py &>/content/logs.txt &


In [14]:
public_url = ngrok.connect("398ykxnqWRpkaexYNblZgOPHLue_6KHcYD3Tm5CmVDHnBtcEg")
print(public_url)


NgrokTunnel: "https://mediaeval-unexpected-alisha.ngrok-free.dev" -> "http://398ykxnqWRpkaexYNblZgOPHLue_6KHcYD3Tm5CmVDHnBtcEg:80"


In [15]:
!pkill streamlit
!pkill ngrok


In [16]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &>/content/streamlit.log &


In [17]:
!grep "Running on" /content/streamlit.log


In [18]:
from pyngrok import ngrok
ngrok.set_auth_token("398ykxnqWRpkaexYNblZgOPHLue_6KHcYD3Tm5CmVDHnBtcEg")


In [19]:
public_url = ngrok.connect(addr=8501, proto="http")
print("Public URL:", public_url)


Public URL: NgrokTunnel: "https://mediaeval-unexpected-alisha.ngrok-free.dev" -> "http://localhost:8501"
